# M1 Notebook 25 — Numerical Stability

**Status:** Runnable first edition

## Learning objectives

- Understand floating-point limits.
- Diagnose cancellation, overflow, and underflow.
- Use stable reformulations.

In [ ]:
from srai_math.utils import environment_info,set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()
from srai_math.numerics import (
    kahan_sum,machine_summary,relative_error,stable_logsumexp,
    stable_quadratic_roots,stable_softmax,
)


## Floating-point environment

In [ ]:
machine_summary()


## Catastrophic cancellation

In [ ]:
x=1e8
naive=np.sqrt(x*x+1)-x
stable=1/(np.sqrt(x*x+1)+x)
{"naive":naive,"stable":stable,"relative_difference":relative_error(naive,stable)}


## Stable quadratic roots

In [ ]:
a,b,c=1.0,1e8,1.0
naive=np.array([(-b+np.sqrt(b*b-4*a*c))/(2*a),(-b-np.sqrt(b*b-4*a*c))/(2*a)])
stable=stable_quadratic_roots(a,b,c)
pd.DataFrame({"naive":np.sort(naive),"stable":np.sort(stable),"reference":np.sort(np.roots([a,b,c]))})


## Overflow-safe log-sum-exp and softmax

In [ ]:
scores=np.array([1000.,1001.,1002.])
{"logsumexp":stable_logsumexp(scores),"softmax":stable_softmax(scores)}


## Summation error

In [ ]:
values=np.concatenate(([1e16],np.ones(100000),[-1e16]))
naive=float(np.sum(values))
stable=kahan_sum(values)
{"naive_sum":naive,"kahan_sum":stable,"expected":100000.0}


## Conditioning versus stability

In [ ]:
epsilons=10.0**(-np.arange(1,13))
condition=[]
solution_error=[]
for e in epsilons:
    A=np.array([[1.,1.],[1.,1.+e]])
    b=np.array([2.,2.+e])
    x=np.linalg.solve(A,b)
    condition.append(np.linalg.cond(A))
    solution_error.append(np.linalg.norm(x-[1,1]))
pd.DataFrame({"epsilon":epsilons,"condition_number":condition,"solution_error":solution_error})


In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
ax.loglog(condition,solution_error,marker="o")
ax.set_xlabel("Condition number"); ax.set_ylabel("Solution error")
ax.set_title("Conditioning and Numerical Error")
plt.show()


## AI interpretation

Stable softmax, log-likelihoods, normalization, and mixed precision are essential in modern machine learning.

## Decision Intelligence case

Small numerical errors can change rankings when alternatives are nearly tied.

In [ ]:
scores=np.array([0.5000000001,0.5,0.4999999999])
rounded=np.round(scores,8)
pd.DataFrame({"raw_score":scores,"rounded_score":rounded},index=["Option_A","Option_B","Option_C"])


## Engineering notes

Scale inputs, avoid unstable subtraction, use logarithmic identities, inspect condition numbers, and test sensitivity.

## Key insight

Numerical stability determines whether mathematically correct formulas produce trustworthy computational results.